# 10 — Nested orders and item-level analytics

**Student exercise**

Complete the tasks in order. Each task includes an expected result, three hints, and a check. All sample data and setup code are included in this notebook.

## 1. Spark configuration

Start a fresh kernel. Use Spark 3.5.x / Scala 2.12 and Java 17. Set `STUDENT_ID` to a lowercase name containing letters, numbers or underscores. Set `SPARK_MASTER=local[2]` for a local Spark session.

SeaweedFS S3 endpoint: `http://localhost:9001`. Use an existing `datalake` bucket. Setup and write tasks overwrite only the specified exercise prefix; use a distinct student ID. The driver and workers must be able to reach this endpoint. Hadoop AWS 3.3.4 must match the Hadoop version in Spark; connector JARs are downloaded from Maven.

In [ ]:
import os
import socket
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

master_url = os.environ.get("SPARK_MASTER", f"spark://{socket.gethostname()}:7077")
STUDENT_ID = "student01"
assert STUDENT_ID and STUDENT_ID[0].islower() and all(c in "abcdefghijklmnopqrstuvwxyz0123456789_" for c in STUDENT_ID)
base_path = f"s3a://datalake/exercises/{STUDENT_ID}/ex10"

spark = (
    SparkSession.builder
    .appName("Exercise-10")
    .master(master_url)
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9001")
    .config("spark.hadoop.fs.s3a.access.key", "team")
    .config("spark.hadoop.fs.s3a.secret.key", "team1234")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.redaction.regex", "(?i)secret|password|token|access[.]?key|credential")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## 2. Sample data

Run this cell, inspect the records, and use the supplied variables in the tasks.

In [ ]:
nested_uri = f"{base_path}/input/nested_orders"
json_rows = [
    '{"order_id":101,"customer":{"id":1,"name":"Asha"},"items":[{"sku":"MUG","qty":2,"price":20.0},{"sku":"BOOK","qty":1,"price":10.0}]}',
    '{"order_id":102,"customer":{"id":2,"name":"Ben"},"items":[{"sku":"BAG","qty":1,"price":50.0}]}',
    '{"order_id":103,"customer":{"id":3,"name":"Carla"},"items":[]}'
]
schema = "order_id INT, customer STRUCT<id:INT,name:STRING>, items ARRAY<STRUCT<sku:STRING,qty:INT,price:DOUBLE>>"
sample_df = spark.read.schema(schema).json(spark.sparkContext.parallelize(json_rows))
sample_df.write.mode("overwrite").json(nested_uri)
nested = spark.read.schema(schema).json(nested_uri)
nested.printSchema()
nested.show(truncate=False)

## 3. Result checks

This small function checks output values and duplicate counts. It is included here so the notebook can run on its own.

In [ ]:
def check_frame(df, columns, expected):
    assert df is not None, "Complete the task before running the check"
    assert set(columns).issubset(df.columns), "Missing output columns"
    def normalize(row):
        return tuple(round(value, 6) if isinstance(value, float) else value for value in row)
    actual = sorted(repr(normalize(row)) for row in df.select(*columns).collect())
    wanted = sorted(repr(normalize(row)) for row in expected)
    assert actual == wanted, f"Expected {wanted}, got {actual}"
    print("PASS:", columns)

## Task 1: Flatten order items

Use explode to produce order_id,customer_name,sku,qty,price; empty items produce no rows.

**Expected result:** Three item rows from two nonempty orders.

In [ ]:
items=None  # TODO: explode and nested-field selections

**Hint 1:** A struct field can be selected with dot notation; explode emits one row per array element.

**Hint 2:** Alias the exploded struct, then select item.sku/item.qty/item.price.

**Hint 3:** nested.select(...,F.explode("items").alias("item")).select(...).

### Check

In [ ]:
check_frame(items,['order_id','sku','qty','price'],[(101,'MUG',2,20.0),(101,'BOOK',1,10.0),(102,'BAG',1,50.0)])

## Task 2: Aggregate totals with SQL

Register item_lines and calculate totals by order_id using SUM(qty*price).

**Expected result:** Both populated orders total50.

In [ ]:
totals=None  # TODO: temporary view and aggregate SQL

**Hint 1:** Aggregate at order grain after exploding item grain.

**Hint 2:** An order with an empty item array has no item total unless deliberately left-joined later.

**Hint 3:** SELECT order_id,SUM(qty*price) AS total ... GROUP BY order_id.

### Check

In [ ]:
check_frame(totals,['order_id','total'],[(101,50.0),(102,50.0)])

## Task 3: Preserve empty arrays and parse JSON text

Create outer_items with explode_outer. Parse json_text into a struct and return parsed.id/name.

**Expected result:** Four outer rows and parsed record7/Gia.

In [ ]:
outer_items=None
parsed=None  # TODO: explode_outer and from_json

**Hint 1:** explode_outer retains one null item for the empty-array order.

**Hint 2:** from_json converts a STRING column using a supplied schema; reading a JSON file is a separate operation.

**Hint 3:** select(from_json(...).alias("record")).select("record.*").

### Check

In [ ]:
check_frame(outer_items,['order_id'],[(101,),(101,),(102,),(103,)])
check_frame(parsed,['id','name'],[(7,'Gia')])

## More practice

Split a MovieLens genre string and explode it; compare movie-grain counts with genre-grain counts.

## Common mistakes

Exploding changes grain and can multiply order-level amounts. from_json requires an explicit schema.

## Finish

Stop Spark before starting another notebook. For write exercises, run setup again before repeating append tasks. S3 data remains available after stopping Spark.

In [ ]:
spark.stop()